# Preparação dos dados - Dataset Adverse Events Drugs API Open FDA

In [26]:
import json
import pandas as pd
import numpy as np

path = "../datasets/drug-event-0001-of-0028-2025-q1.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

reports_raw = data["results"]

In [28]:
reports = pd.json_normalize(reports_raw)

reports.shape

(12000, 40)

In [29]:
cols = [
    "safetyreportid",
    "serious",
    "receivedate",
    "receiptdate",
    "occurcountry",
    "primarysource.reportercountry",
    "primarysource.qualification",
    "patient.patientsex",
    "patient.patientonsetage",
    "patient.patientonsetageunit",
    "patient.patientweight",
]

df = reports[[c for c in cols if c in reports.columns]].copy()

In [ ]:
# Não usar para treinamento
leakage_cols = [
    "seriousnessdeath",
    "seriousnesslifethreatening",
    "seriousnesshospitalization",
    "seriousnessdisabling",
    "seriousnesscongenitalanomali",
    "seriousnessother",
]

In [30]:
df = df[df["serious"].isin(["1", "2"])].copy()

df["target_serious"] = df["serious"].map({
    "1": 1,
    "2": 0,
})

df = df.drop(columns=["serious"])

In [31]:
df["target_serious"].value_counts(normalize=True)

target_serious
1    0.632333
0    0.367667
Name: proportion, dtype: float64

In [32]:
df_drugs = pd.json_normalize(
    reports_raw,
    record_path=["patient", "drug"],
    meta=["safetyreportid"],
    errors="ignore"
)

drug_counts = (
    df_drugs
    .groupby("safetyreportid")
    .size()
    .reset_index(name="drug_count")
)

df = df.merge(drug_counts, on="safetyreportid", how="left")

df["drug_count"] = df["drug_count"].fillna(0).astype(int)

In [33]:
df["patient.patientonsetage"] = pd.to_numeric(
    df["patient.patientonsetage"],
    errors="coerce"
)

In [34]:
def convert_age_to_years(age, unit):
    if pd.isna(age) or pd.isna(unit):
        return np.nan

    unit = str(unit)

    if unit == "800":      # decade
        return age * 10
    elif unit == "801":    # year
        return age
    elif unit == "802":    # month
        return age / 12
    elif unit == "803":    # week
        return age / 52
    elif unit == "804":    # day
        return age / 365
    elif unit == "805":    # hour
        return age / (365 * 24)
    else:
        return np.nan

df["age_years"] = df.apply(
    lambda row: convert_age_to_years(
        row.get("patient.patientonsetage"),
        row.get("patient.patientonsetageunit")
    ),
    axis=1
)

In [35]:
df.loc[(df["age_years"] < 0) | (df["age_years"] > 120), "age_years"] = np.nan

In [36]:
df["age_group"] = pd.cut(
    df["age_years"],
    bins=[0, 18, 40, 60, 120],
    labels=["0-18", "18-40", "40-60", "60+"],
    include_lowest=True
)

In [38]:
df["age_missing"] = df["age_years"].isna().astype(int)

In [39]:
df["sex"] = df["patient.patientsex"].map({
    "1": "male",
    "2": "female",
    "0": "unknown"
}).fillna("unknown")

In [40]:
df["weight_kg"] = pd.to_numeric(
    df["patient.patientweight"],
    errors="coerce"
)

df.loc[(df["weight_kg"] < 1) | (df["weight_kg"] > 300), "weight_kg"] = np.nan

df["weight_missing"] = df["weight_kg"].isna().astype(int)

In [41]:
df["weight_kg"] = df["weight_kg"].fillna(df["weight_kg"].median())

In [42]:
df_reactions = pd.json_normalize(
    reports_raw,
    record_path=["patient", "reaction"],
    meta=["safetyreportid"],
    errors="ignore"
)

In [43]:
reaction_counts = (
    df_reactions
    .groupby("safetyreportid")
    .size()
    .reset_index(name="reaction_count")
)

df = df.merge(reaction_counts, on="safetyreportid", how="left")
df["reaction_count"] = df["reaction_count"].fillna(0).astype(int)

In [45]:
df["receivedate"] = pd.to_datetime(
    df["receivedate"],
    errors="coerce",
    format="%Y%m%d"
)

df["received_year"] = df["receivedate"].dt.year
df["received_month"] = df["receivedate"].dt.month

In [46]:
features = [
    "drug_count",
    "reaction_count",
    "age_years",
    "age_missing",
    "weight_kg",
    "weight_missing",
    "sex",
    "age_group",
    "occurcountry",
    "primarysource.reportercountry",
    "primarysource.qualification",
]

In [47]:
features = [c for c in features if c in df.columns]

model_df = df[features + ["target_serious"]].copy()

In [48]:
numeric_features = [
    "drug_count",
    "reaction_count",
    "age_years",
    "age_missing",
    "weight_kg",
    "weight_missing",
    "received_year",
    "received_month",
]

numeric_features = [c for c in numeric_features if c in model_df.columns]

categorical_features = [
    "sex",
    "age_group",
    "occurcountry",
    "primarysource.reportercountry",
    "primarysource.qualification",
]

categorical_features = [c for c in categorical_features if c in model_df.columns]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report, confusion_matrix

Resolved 51 packages in 194ms                                        
Installed 3 packages in 78ms                                     
 + joblib==1.5.3
 + scikit-learn==1.8.0
 + threadpoolctl==3.6.0


In [52]:
X = model_df.drop(columns=["target_serious"])
y = model_df["target_serious"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [53]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [54]:
clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=100,
        random_state=42
    ))
])

clf.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [56]:
y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.78625
Precision: 0.8758414360508602
Recall: 0.7714097496706193
              precision    recall  f1-score   support

           0       0.67      0.81      0.74       882
           1       0.88      0.77      0.82      1518

    accuracy                           0.79      2400
   macro avg       0.77      0.79      0.78      2400
weighted avg       0.80      0.79      0.79      2400

[[ 716  166]
 [ 347 1171]]


In [57]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_leaf=50,
        random_state=42,
        n_jobs=-1
    ))
])

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.72      0.57      0.64       882
           1       0.78      0.87      0.82      1518

    accuracy                           0.76      2400
   macro avg       0.75      0.72      0.73      2400
weighted avg       0.76      0.76      0.75      2400

[[ 504  378]
 [ 198 1320]]
